# Task 6 — Xác nhận dữ liệu đầu vào thực nghiệm

Đối soát **toàn bộ** CSV đã kiểm tra ở Task 4 với Parquet hiện có, sau đó kiểm tra cấu trúc hai tập và thử tạo mảng đầu vào TimesNet/SSA. Notebook chỉ đọc file dữ liệu. Kết quả được ghi vào thư mục báo cáo riêng. Không cần GPU.

Trên Colab, đặt bốn file nguồn trong cùng một thư mục Drive (hoặc sửa từng đường dẫn ở ô cấu hình). Chạy lần lượt từ trên xuống. Trên máy cá nhân, đặt notebook trong cấu trúc thư mục repo hoặc sửa `DATA_DIR` và `TASK4_DIR` cho đúng.

Trong phạm vi kiểm tra này, “khớp toàn bộ” nghĩa là từng giá trị sau khi đọc CSV và đổi sang kiểu dữ liệu tương ứng bằng giá trị Parquet, trên cùng thứ tự khóa của hai file. Với hai cột danh sách, so từng phần tử theo thứ tự giờ. Nếu thứ tự khác, phép kiểm tra dừng và báo lỗi; không kết luận hai file khác nội dung khi chưa đối chiếu lại theo khóa.


In [1]:
from pathlib import Path
from datetime import date, datetime, timezone
import csv
import hashlib
import json
import platform
import sys

ON_COLAB = 'google.colab' in sys.modules
if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    WORKSPACE = Path('/content/drive/MyDrive/PTLDKD')
    DATA_DIR = WORKSPACE / 'data'
    TASK4_DIR = WORKSPACE / 'Task04_Data_Validation/results'
    RESULTS_BASE = WORKSPACE / 'Task06_Data_Preparation/results'
else:
    cwd = Path.cwd().resolve()
    WORKSPACE = next((p for p in (cwd, *cwd.parents)
                      if (p / 'Task04_Data_Validation/results').is_dir()), cwd)
    DATA_DIR = WORKSPACE
    TASK4_DIR = WORKSPACE / 'Task04_Data_Validation/results'
    RESULTS_BASE = WORKSPACE / 'Task06_Data_Preparation/results'

CSV_PATHS = {split: DATA_DIR / f'{split}.csv' for split in ('train', 'eval')}
PARQUET_PATHS = {split: DATA_DIR / f'{split}.parquet' for split in ('train', 'eval')}
TASK4_RUN_INFO = TASK4_DIR / 'run_info.json'
TASK4_SUMMARY = TASK4_DIR / 'validation_summary.csv'
# Hai file Task 4 giúp nối nguồn với lần kiểm định trước; kiểm tra trực tiếp
# toàn bộ dữ liệu vẫn chạy được khi không có hai file này.


In [2]:
import numpy as np
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.csv as pacsv
import pyarrow.parquet as pq

COLS = (
    'city_id', 'store_id', 'management_group_id', 'first_category_id',
    'second_category_id', 'third_category_id', 'product_id', 'dt',
    'sale_amount', 'hours_sale', 'stock_hour6_22_cnt', 'hours_stock_status',
    'discount', 'holiday_flag', 'activity_flag', 'precpt',
    'avg_temperature', 'avg_humidity', 'avg_wind_level',
)
KEY = ('store_id', 'product_id', 'dt')
HOUR_FIELDS = ('hours_sale', 'hours_stock_status')
EXPECTED = {
    'train': {'rows': 4_500_000, 'days': 90, 'start': '2024-03-28', 'end': '2024-06-25'},
    'eval': {'rows': 350_000, 'days': 7, 'start': '2024-06-26', 'end': '2024-07-02'},
}
PAIR_COUNT = 50_000
AUTHOR_COMMIT = '8af694a09a5a427d1d50648d348c022565a781f2'
PROJECT_BASE_COMMIT = 'da2e3ccd7e64bfc944c7b28d9c4c77e975e4fe05'

def require(condition, message):
    if not condition:
        raise ValueError(message)

def fingerprint(path):
    h = hashlib.sha256()
    with path.open('rb') as file:
        for block in iter(lambda: file.read(8 * 1024 * 1024), b''):
            h.update(block)
    return {'path': str(path.resolve()), 'bytes': path.stat().st_size,
            'mtime_ns': path.stat().st_mtime_ns, 'sha256': h.hexdigest()}

def parse_hour_column(col, value_type):
    # CSV chứa mảng dạng [x, y, ...]. Tách và ép kiểu bằng Arrow.
    bracketed = pc.and_(pc.starts_with(col, '['), pc.ends_with(col, ']'))
    require(col.null_count == 0 and pc.all(bracketed).as_py(), 'CSV có mảng giờ sai định dạng.')
    strings = pc.split_pattern(pc.utf8_slice_codeunits(col, 1, -1), ',')
    require(pc.all(pc.equal(pc.list_value_length(strings), 24)).as_py(),
            'CSV có mảng giờ không đủ 24 phần tử.')
    values = pc.cast(pc.utf8_trim_whitespace(pc.list_flatten(strings)), value_type)
    require(values.null_count == 0, 'CSV có phần tử giờ rỗng.')
    return pa.FixedSizeListArray.from_arrays(values, 24)

def compare_split(csv_path, parquet_path, split):
    spec = EXPECTED[split]
    with pq.ParquetFile(parquet_path) as pf:
        schema = pf.schema_arrow
        require(tuple(schema.names) == COLS, f'{split}: Parquet không đúng danh sách 19 cột.')
        require(pf.metadata.num_rows == spec['rows'], f'{split}: sai số dòng Parquet.')
        require(pa.types.is_string(schema.field('dt').type), f'{split}: dt phải là chuỗi.')
        for col in HOUR_FIELDS:
            field_type = schema.field(col).type
            require(pa.types.is_list(field_type) or pa.types.is_large_list(field_type)
                    or pa.types.is_fixed_size_list(field_type), f'{col} phải là kiểu danh sách.')
        require(pa.types.is_floating(schema.field('hours_sale').type.value_type),
                'hours_sale phải chứa số thực.')
        require(pa.types.is_integer(schema.field('hours_stock_status').type.value_type),
                'hours_stock_status phải chứa số nguyên.')

        csv_types = {field.name: pa.string() if field.name in HOUR_FIELDS else field.type
                     for field in schema}
        parquet_batches = iter(pf.iter_batches(batch_size=65_536, columns=list(COLS)))
        pending = next(parquet_batches, None)
        offset = 0
        compared = 0
        discount_review = 0
        with pacsv.open_csv(
            csv_path, read_options=pacsv.ReadOptions(block_size=16 * 1024 * 1024),
            convert_options=pacsv.ConvertOptions(column_types=csv_types)
        ) as reader:
            require(tuple(reader.schema.names) == COLS, f'{split}: CSV không đúng danh sách 19 cột.')
            for csv_batch in reader:
                for col in COLS:
                    require(csv_batch[col].null_count == 0, f'{split}: CSV có null ở {col}.')
                parsed = {}
                for col in HOUR_FIELDS:
                    parsed[col] = parse_hour_column(csv_batch[col], schema.field(col).type.value_type)
                start = 0
                while start < len(csv_batch):
                    require(pending is not None, f'{split}: CSV có nhiều dòng hơn Parquet.')
                    size = min(len(csv_batch) - start, len(pending) - offset)
                    for col in COLS:
                        right = pending[col].slice(offset, size)
                        require(right.null_count == 0, f'{split}: Parquet có null ở {col}.')
                        if col in HOUR_FIELDS:
                            require(pc.all(pc.equal(pc.list_value_length(right), 24)).as_py(),
                                    f'{split}: Parquet có mảng {col} không đủ 24 phần tử.')
                            right = pc.list_flatten(right)
                            left = pc.list_flatten(parsed[col].slice(start, size))
                            require(right.null_count == 0, f'{split}: phần tử giờ null ở {col}.')
                        else:
                            left = csv_batch[col].slice(start, size)
                        require(left.equals(right),
                                f'{split}: khác dữ liệu hoặc thứ tự ở {col}, '
                                f'dòng {compared+1:,}–{compared+size:,}.')
                    start += size
                    offset += size
                    compared += size
                    if offset == len(pending):
                        pending = next(parquet_batches, None)
                        offset = 0
                review_mask = pc.or_(pc.less(csv_batch['discount'], 0),
                                     pc.greater(csv_batch['discount'], 1))
                discount_review += int(pc.sum(pc.cast(review_mask, pa.int64())).as_py())

        require(pending is None and compared == spec['rows'], f'{split}: không khớp tổng số dòng.')
        # Đọc đúng 3 cột khóa để kiểm tra lịch sử và ngày; không tải mảng giờ lần nữa.
        k = pf.read(columns=list(KEY))

    sid = k['store_id'].to_numpy()
    pid = k['product_id'].to_numpy()
    day = pc.cast(pc.strptime(k['dt'], format='%Y-%m-%d', unit='s'), pa.date32())
    day = pc.cast(day, pa.int32()).to_numpy()
    order = np.lexsort((day, pid, sid))
    sid, pid, day = sid[order], pid[order], day[order]
    same_pair = (sid[1:] == sid[:-1]) & (pid[1:] == pid[:-1])
    starts = np.r_[0, np.flatnonzero(~same_pair) + 1]
    stops = np.r_[starts[1:], len(day)]
    duplicate_keys = int(np.count_nonzero(same_pair & (day[1:] == day[:-1])))
    # Với khóa không trùng, đúng số ngày và đúng hai mốc là đủ để chứng minh liên tục.
    start_day = (date.fromisoformat(spec['start']) - date(1970, 1, 1)).days
    end_day = (date.fromisoformat(spec['end']) - date(1970, 1, 1)).days
    incomplete = int(np.count_nonzero((stops - starts != spec['days'])
                     | (day[starts] != start_day) | (day[stops - 1] != end_day)))
    require(len(starts) == PAIR_COUNT and duplicate_keys == 0 and incomplete == 0,
            f'{split}: cặp hoặc ngày không đúng kỳ vọng.')
    min_date = str(np.datetime64(int(day.min()), 'D'))
    max_date = str(np.datetime64(int(day.max()), 'D'))
    summary = {
        'split': split, 'parquet_path': str(parquet_path.resolve()),
        'columns': len(schema), 'rows': compared, 'pairs': len(starts),
        'date_start': min_date, 'date_end': max_date,
        'duplicate_keys': duplicate_keys, 'incomplete_pairs': incomplete,
        'invalid_24_hour_arrays': 0, 'discount_review_rows': discount_review,
        'csv_parquet_mismatched_rows': 0, 'scope': 'full', 'status': 'PASS',
    }
    return summary, np.column_stack((sid[starts], pid[starts]))


## Đối chiếu toàn bộ và kiểm tra cấu trúc

Đọc theo từng lô để giảm RAM. So sánh 17 cột đơn và 2 cột danh sách giờ. Sau đó sắp xếp khóa trong bộ nhớ để kiểm tra số chuỗi, ngày bị thiếu và khóa trùng. File đầu vào không cần được lưu sẵn theo thứ tự sắp xếp.


In [3]:
task4_meta = json.loads(TASK4_RUN_INFO.read_text(encoding='utf-8-sig')) if TASK4_RUN_INFO.is_file() else {}
task4_alerts = {}
if TASK4_SUMMARY.is_file():
    with TASK4_SUMMARY.open(encoding='utf-8-sig', newline='') as f:
        for entry in csv.DictReader(f):
            if entry['check'] == 'discount_outside_0_1':
                task4_alerts[entry['split']] = int(entry['violations'])

summaries, pair_keys, source_info = {}, {}, {}
for split in EXPECTED:
    path_csv, path_pq = CSV_PATHS[split], PARQUET_PATHS[split]
    require(path_csv.is_file() and path_pq.is_file(),
            f'Thiếu file cho {split}: {path_csv} hoặc {path_pq}. Sửa đường dẫn ở ô cấu hình.')
    print(f'Đang đối chiếu toàn bộ {split}...', flush=True)
    before_csv = (path_csv.stat().st_size, path_csv.stat().st_mtime_ns)
    before_pq = (path_pq.stat().st_size, path_pq.stat().st_mtime_ns)
    summaries[split], pair_keys[split] = compare_split(path_csv, path_pq, split)
    source_info[split] = {'csv': fingerprint(path_csv), 'parquet': fingerprint(path_pq)}
    require(before_csv == (path_csv.stat().st_size, path_csv.stat().st_mtime_ns)
            and before_pq == (path_pq.stat().st_size, path_pq.stat().st_mtime_ns),
            f'{split}: file nguồn đổi trong lúc kiểm tra.')
    prior = task4_meta.get('sources', {}).get(split)
    now = source_info[split]['csv']
    if prior:
        source_info[split]['task4_size_matches'] = now['bytes'] == prior.get('bytes')
        source_info[split]['task4_mtime_matches'] = now['mtime_ns'] == prior.get('mtime_ns')
    if split in task4_alerts:
        require(summaries[split]['discount_review_rows'] == task4_alerts[split],
                f'{split}: số dòng discount REVIEW không khớp Task 4.')
    item = summaries[split]
    print(f"{split}: {item['rows']:,} dòng × {item['columns']} cột; "
          f"{item['pairs']:,} cặp; {item['date_start']} → {item['date_end']}; "
          f"discount REVIEW: {item['discount_review_rows']}; PASS")

require(np.array_equal(pair_keys['train'], pair_keys['eval']),
        'Hai tập không có cùng tập cặp cửa hàng–sản phẩm.')
require(summaries['train']['date_end'] < summaries['eval']['date_start'],
        'Khoảng ngày train/eval chồng lặp.')
print('Đối chiếu train/eval: cùng 50.000 cặp, không trùng khóa ngày.')


Đang đối chiếu toàn bộ train...
train: 4,500,000 dòng × 19 cột; 50,000 cặp; 2024-03-28 → 2024-06-25; discount REVIEW: 34; PASS
Đang đối chiếu toàn bộ eval...
eval: 350,000 dòng × 19 cột; 50,000 cặp; 2024-06-26 → 2024-07-02; discount REVIEW: 0; PASS
Đối chiếu train/eval: cùng 50.000 cặp, không trùng khóa ngày.


## Chạy thử cấu trúc đầu vào hai mô hình

Chọn hai cặp theo thứ tự khóa. Tạo mảng TimesNet trên train theo cách của mã tác giả: chia 90 ngày thành ba đoạn 30 ngày, lấy giờ 06:00 đến trước 22:00 và đánh dấu giờ hết hàng bằng NaN ở kênh doanh số. Bốn biến bổ trợ được chuẩn hóa theo từng đoạn, rồi ghép chỉ số giờ chuẩn hóa.

Với SSA, kiểm tra mẫu ghép đủ 90 ngày train và 7 ngày eval theo mỗi cặp, rồi kiểm tra phép reshape ba cột số mà SSA dùng. Chỉ kiểm tra đầu vào, chưa chạy dự báo.


In [4]:
example_pairs = pair_keys['train'][:2]
sample_filter = [[('store_id', '=', int(s)), ('product_id', '=', int(p))]
                 for s, p in example_pairs]
tr = pq.read_table(PARQUET_PATHS['train'], filters=sample_filter).sort_by([(k, 'ascending') for k in KEY])
ev = pq.read_table(PARQUET_PATHS['eval'], filters=sample_filter).sort_by([(k, 'ascending') for k in KEY])
require(len(tr) == 180 and len(ev) == 14, 'Thiếu dòng ở hai cặp mẫu.')

hour_sales = np.array(tr['hours_sale'].to_pylist(), dtype=np.float64)
hour_stock = np.array(tr['hours_stock_status'].to_pylist(), dtype=np.int64)
sales = hour_sales.reshape(6, 30, 24)[..., 6:22]
mask = hour_stock.reshape(6, 30, 24)[..., 6:22] == 1
sales = np.where(mask, np.nan, sales)
cov = np.column_stack([tr[c].to_numpy() for c in
                       ('discount', 'holiday_flag', 'precpt', 'avg_temperature')]).reshape(6, 30, 4)
cov = cov / (cov.max(axis=1, keepdims=True) + 0.1)
features = np.concatenate([
    sales[..., None],
    np.broadcast_to(cov[:, :, None, :], (6, 30, 16, 4)),
    np.broadcast_to(np.arange(16)[None, None, :, None] / 15, (6, 30, 16, 1)),
], axis=-1).reshape(6, 480, 6)
require(features.shape == (6, 480, 6), 'Kích thước đầu vào TimesNet không đúng.')
require(np.array_equal(np.isnan(features[:, :, 0]), mask.reshape(6, 480)),
        'Mặt nạ hết hàng ở mẫu không khớp.')
require(np.isfinite(features[:, :, 1:]).all(), 'Đặc trưng bổ trợ không hữu hạn.')

combined = pa.concat_tables([tr.select(['store_id','product_id','dt','discount','sale_amount']),
                             ev.select(['store_id','product_id','dt','discount','sale_amount'])])
combined = combined.sort_by([(k, 'ascending') for k in KEY])
dts = np.array(combined['dt'].to_pylist(), dtype='datetime64[D]')
full_days = np.arange(np.datetime64(EXPECTED['train']['start']),
                      np.datetime64(EXPECTED['eval']['end']) + np.timedelta64(1, 'D'))
require(np.array_equal(dts.reshape(2, 97), np.tile(full_days, (2, 1))),
        'Mỗi cặp SSA phải đủ 97 ngày theo thứ tự.')
for index, pair in enumerate(example_pairs):
    i = slice(index*97, (index+1)*97)
    require((combined['store_id'].to_numpy()[i] == pair[0]).all()
            and (combined['product_id'].to_numpy()[i] == pair[1]).all(),
            'Một khối 97 ngày có khóa không đồng nhất.')
ssa = np.column_stack(((dts - full_days[0]).astype(int),
                       combined['discount'].to_numpy(),
                       combined['sale_amount'].to_numpy())).reshape(2, 97, 3)
sample_check = {
    'pairs': example_pairs.tolist(),
    'timesnet_shape': list(features.shape),
    'timesnet_masked_hours': int(mask.sum()),
    'ssa_continuous_shape': list(ssa.shape),
    'ssa_train_days': 90,
    'ssa_eval_days': 7,
}
print(f'Cặp mẫu: {sample_check["pairs"]}')
print(f'TimesNet: {features.shape}, {mask.sum()} giờ có mặt nạ hết hàng.')
print(f'SSA: {ssa.shape}, mỗi cặp 90 ngày train + 7 ngày eval.')


Cặp mẫu: [[0, 4], [0, 6]]
TimesNet: (6, 480, 6), 586 giờ có mặt nạ hết hàng.
SSA: (2, 97, 3), mỗi cặp 90 ngày train + 7 ngày eval.


## Lưu sản phẩm bàn giao

Mỗi lần chạy tạo thư mục riêng. File `input_check.csv` có hai dòng; `run_info.md` ghi kết luận, nguồn và SHA-256. SHA-256 dùng để nhận diện đúng file ở lần sử dụng sau. Task 4 chỉ ghi dung lượng và thời điểm sửa, không có SHA-256 để so với lần trước.


In [5]:
run_time = datetime.now(timezone.utc)
run_dir = RESULTS_BASE / run_time.strftime('%Y%m%dT%H%M%S%fZ')
run_dir.mkdir(parents=True, exist_ok=False)
rows = [summaries['train'], summaries['eval']]
with (run_dir / 'input_check.csv').open('w', encoding='utf-8-sig', newline='') as file:
    out = csv.DictWriter(file, fieldnames=list(rows[0]))
    out.writeheader()
    out.writerows(rows)

metadata = {
    'run_utc': run_time.isoformat(),
    'method': 'full_row_aligned_exact_csv_vs_parquet_after_type_conversion',
    'project_base_commit': PROJECT_BASE_COMMIT,
    'author_commit': AUTHOR_COMMIT,
    'python': platform.python_version(), 'numpy': np.__version__, 'pyarrow': pa.__version__,
    'sources': source_info, 'summary': rows, 'sample_check': sample_check,
    'task4_run_info': str(TASK4_RUN_INFO) if task4_meta else None,
    'task4_summary': str(TASK4_SUMMARY) if task4_alerts else None,
}
(run_dir / 'run_info.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

fmt = lambda n: f'{n:,}'.replace(',', '.')
lines = [
    '# Task 6 — Kết quả xác nhận đầu vào', '',
    '**Kết luận: dùng lại hai file Parquet hiện có.** Đã đối chiếu đầy đủ '
    f"{fmt(sum(x['rows'] for x in rows))} bản ghi × 19 cột với CSV, "
    'bao gồm từng phần tử trong hai danh sách 24 giờ. Không có khác biệt '
    'sau khi đọc và chuyển kiểu theo schema Parquet.', '',
    '| Tập | Số dòng | Số cặp | Khoảng ngày | Khóa trùng | Cặp thiếu ngày | Mảng sai độ dài | Discount ngoài [0, 1] |',
    '|---|---:|---:|---|---:|---:|---:|---:|',
]
for row in rows:
    lines.append(f"| {row['split']} | {fmt(row['rows'])} | {fmt(row['pairs'])} | "
                 f"{row['date_start']} đến {row['date_end']} | {row['duplicate_keys']} | "
                 f"{row['incomplete_pairs']} | {row['invalid_24_hour_arrays']} | "
                 f"{row['discount_review_rows']} |")
lines += [
    '',
    'Hai tập có cùng danh sách cặp và các khoảng ngày không giao nhau. '
    'Giữ nguyên tất cả giá trị discount đã được đánh dấu REVIEW.',
    '',
    f"- Mẫu TimesNet: {tuple(features.shape)}, 6 đoạn 30 ngày từ 2 cặp; "
    f'{int(mask.sum())} giờ được đánh dấu NaN ở kênh doanh số.',
    f'- Mẫu SSA: {tuple(ssa.shape)}, mỗi cặp gồm 90 ngày train và 7 ngày eval.',
    '- Sắp xếp khóa trong bộ nhớ trước khi tạo mảng theo quy trình repo. '
    'Không ghi đè, chuyển đổi lại hoặc sao chép dữ liệu gốc.',
    '- Task 6 dừng ở kiểm tra dữ liệu và kích thước; chưa huấn luyện hay dự báo.',
    '',
    '## Nguồn và cách chạy lại', '',
    f'- Lượt kiểm tra (UTC): {run_time.isoformat()}. Python {platform.python_version()}, '
    f'NumPy {np.__version__}, PyArrow {pa.__version__}.',
    f'- Repo nhóm đối chiếu: {PROJECT_BASE_COMMIT}. Mã tác giả đối chiếu: {AUTHOR_COMMIT}.',
    '- Mở 06_prepare_data.ipynb, chỉnh đường dẫn nếu cần, rồi chạy tất cả các ô. '
    'Đối soát theo lô; thứ tự dòng CSV và Parquet được kiểm tra qua từng khóa và giá trị.',
]
if task4_meta:
    lines.append(f'- Metadata Task 4: {TASK4_RUN_INFO}. Kích thước và mtime đối chiếu ở dưới; '
                 'mtime có thể đổi khi sao chép file giữa các môi trường.')
if task4_alerts:
    lines.append(f'- Bảng Task 4: {TASK4_SUMMARY}. Số dòng discount REVIEW khớp ở cả hai tập.')
for split, info in source_info.items():
    lines += ['', f'### {split}', '']
    for kind in ('csv', 'parquet'):
        f = info[kind]
        lines += [f"- {kind.upper()}: `{f['path']}` ({fmt(f['bytes'])} byte).",
                  f"  SHA-256: `{f['sha256']}`."]
    if 'task4_size_matches' in info:
        lines.append(f"- CSV so với metadata Task 4: khớp dung lượng = "
                     f"{info['task4_size_matches']}; khớp mtime = {info['task4_mtime_matches']}.")
lines += ['', 'Đường dẫn nêu trên là đường dẫn của máy hoặc phiên chạy này. '
          'Nếu dùng Drive, chọn hai bản có SHA-256 tương ứng hoặc chạy lại đối soát với dữ liệu trên Drive.']
(run_dir / 'run_info.md').write_text('\n'.join(lines) + '\n', encoding='utf-8')
print(f'Đã lưu kết quả: {run_dir}')
print('input_check.csv • run_info.md • run_info.json')


Đã lưu kết quả: Task06_Data_Preparation/results/20261113T082302850933Z
input_check.csv • run_info.md • run_info.json
